# Lab 7 — Is the model even identifiable? A symbolic structural-identifiability check

*Sits between Lab 6 (control theory on cellular dynamics) and Lab 8 (the anatomical compiler).*

Before you **fit** a mechanistic model — the Hill-ODE circuits of Lab 1, the IRMA-topology GRN in
`jaxctrl/examples/irma_sindy_lqr.ipynb`, or the linear surrogate that warm-starts the anatomical
compiler in `scripts/benchmark_anatomical_compiler.py` — there is a question worth asking *first*:
**can its parameters even be recovered from what you can measure, in the ideal limit of perfect,
noise-free data?** If the answer is no, no quantity of data fixes it; the best fit will sit in a flat
valley and the simulation-based-inference posterior of Lab 8 will be (correctly) ridge-shaped. The
fix is then not "more data" but "measure more species" or "re-parameterize."

This is **structural identifiability** — a property of the *model structure*, decided **symbolically,
a priori**. It is a different question from two neighbours you'll meet elsewhere in this course:

| | asks | decided by | where |
|---|---|---|---|
| **module** identifiability | does the *network topology* split into stable, distinct modules? | the Hodge-Laplacian spectral gap | Lab 4 (the Module Identifiability Index) |
| **structural** identifiability *(this lab)* | are the model's rate constants / Hill coefficients uniquely fixed by the outputs, with perfect data? | differential algebra / Taylor-series rank — symbolic | here; tools: COMBOS, SIAN, DAISY, StructuralIdentifiability.jl |
| **practical** identifiability | with *finite, noisy* data, how well-constrained is each parameter? | profile likelihoods / Fisher info / posterior width | Lab 8 (the SBI / CellFlow side) |

We implement the **Taylor-series / observability-rank** version in `sympy` — the symbolic counterpart
of `jaxctrl.observability_matrix` / `is_observable` (which do the *linear*, numeric version, §5 below).
It is small enough to read in one screen and big enough to handle real gene circuits; for free Hill
exponents or larger models, hand off to **COMBOS** (the DiStefano-lab differential-algebra tool —
[biocyb0.cs.ucla.edu](http://biocyb0.cs.ucla.edu/wp/)) or one of the other dedicated tools.

**Needs:** `sympy`, `numpy`, `scipy`, `jaxctrl` (a `uv sync` of this repo gives you all of them).

In [1]:
import sys, json
from pathlib import Path
import numpy as np
import sympy as sp
import jax.numpy as jnp
import jaxctrl


def structural_identifiability(f, g, x, theta, *, x0_known=False, order=None,
                               mode="numeric", seed=0):
    """Local structural identifiability via the observability / Taylor-series rank condition.

    Model:  dx/dt = f(x; theta),  y = g(x; theta).   Treat the unknown parameters as states
    with d(theta)/dt = 0; let  z = x ++ theta  and  F(z) = f ++ 0.   The "observation algebra"
    is generated by  g, L_F g, L_F^2 g, ...  where  L_F h = sum_i (dh/dz_i) F_i  (the Lie
    derivative of h along F).   The model is *locally structurally identifiable at a generic
    point z* iff

        rank  O  =  dim z          (or, if the initial state x0 is known: = dim theta)

    where  O = [ d/dz of (g, L_F g, ..., L_F^N g) ]  stacked over all outputs, and N >= dim z - 1
    is generically sufficient.   If rank-deficient, the (right) nullspace of O gives the
    directions in (x0, theta)-space that no output derivative can resolve — the *unidentifiable
    combinations*; their projection onto the theta-block is what tools like COMBOS report as
    "identifiable parameter combinations".

    Parameters
    ----------
    f, g : lists of sympy expressions (the RHS of dx/dt, and the outputs y)
    x, theta : lists of sympy symbols (states; unknown parameters)
    x0_known : if True, the initial state is assumed known (test parameters only)
    order : number of Lie derivatives to stack (default dim z - 1 — the sufficient bound)
    mode : "symbolic" (exact sp.Matrix.rank — fine for tiny rational models) or
           "numeric" (substitute distinct random integers, take the numeric rank — equal to
           the generic symbolic rank with probability 1; the scalable workhorse)

    Returns
    -------
    dict(rank, full, identifiable, null, params)
    """
    x, theta = list(x), list(theta)
    z = x + theta
    F = list(f) + [sp.Integer(0)] * len(theta)
    Lf = lambda h: sum(sp.diff(h, zi) * Fi for zi, Fi in zip(z, F))
    N = order if order is not None else max(len(z) - 1, 1)
    rows = []
    for gj in g:
        h = sp.cancel(sp.together(gj))
        for _ in range(N + 1):
            rows.append([sp.diff(h, zi) for zi in z])
            h = sp.cancel(sp.together(Lf(h)))
    O = sp.Matrix(rows)
    cols = list(range(len(x), len(z))) if x0_known else list(range(len(z)))
    target = len(cols)
    Osub = O[:, cols]
    if mode == "symbolic":
        rank = int(Osub.rank())
        null = [list(v) for v in (Osub.nullspace() if rank < target else [])]
    else:
        rng = np.random.default_rng(seed)
        repl = {zi: sp.Rational(int(pr))
                for zi, pr in zip(z, rng.choice(range(2, 500), size=len(z), replace=False))}
        Onum = np.array([[float(e.subs(repl)) for e in r] for r in Osub.tolist()], dtype=float)
        rank = int(np.linalg.matrix_rank(Onum))
        from scipy.linalg import null_space
        null = (null_space(Onum).T.round(3).tolist() if rank < target else [])
    return dict(rank=rank, full=int(target), identifiable=(rank == int(target)),
                null=null, params=([str(t) for t in theta] if x0_known
                                   else [str(s) for s in x + theta]))


def report(name, r):
    """Pretty-print one structural-identifiability result."""
    tag = "IDENTIFIABLE" if r["identifiable"] else "NOT identifiable"
    extra = ""
    if not r["identifiable"] and r["null"]:
        ndof = len(r["null"])
        is_exact_1d = (ndof == 1 and all(float(c) == round(float(c)) for c in r["null"][0]))
        if is_exact_1d:
            terms = " ".join(f"{float(c):+.0f}*{p}" for c, p in zip(r["null"][0], r["params"]) if abs(float(c)) > 1e-9)
            extra = f"   |  unresolved combination:  {terms}   (only the complementary direction is pinned down)"
        else:
            extra = f"   |  {ndof} unresolved d.o.f. among {{{', '.join(r['params'])}}}  (only {r['rank']} independent combinations are determined)"
    print(f"{name:<48s} rank {r['rank']}/{r['full']}   {tag}{extra}")
    return r
print("ready")

ready


## 1. The method, in one paragraph

Write the model as $\dot x = f(x;\theta)$, $y = g(x;\theta)$. Treat the unknown parameters as
constant states ($\dot\theta = 0$): let $z = (x,\theta)$, $F(z) = (f, 0)$. Differentiating the
output and substituting the dynamics, the Taylor coefficients of $y$ at $t=0$ are
$y^{(k)}(0) = (L_F^k g)(z_0)$, where $L_F h = \sum_i (\partial h/\partial z_i)\,F_i$ is the Lie
derivative. So the entire information that the input–output map carries about $z_0$ lives in the
list $\{g, L_F g, L_F^2 g, \dots\}$ — and the model is **locally structurally identifiable at a
generic $z_0$** iff
$$
\operatorname{rank}\,\mathcal O(z_0) = \dim z,\qquad
\mathcal O = \big[\,\nabla_z g;\ \nabla_z L_F g;\ \dots;\ \nabla_z L_F^{N} g\,\big],\quad N \ge \dim z - 1.
$$
If the initial state is known, restrict $\mathcal O$ to the $\theta$-columns and ask for rank
$=\dim\theta$. If $\mathcal O$ is rank-deficient, its (right) nullspace names the directions in
$z$-space that no output derivative can move — the *unidentifiable combinations*; everything
orthogonal to them is fine. Computing $\operatorname{rank}\mathcal O$ symbolically is exact but
can blow up; substituting distinct random integers for the symbols and taking the *numeric* rank
gives the same generic rank with probability 1 — that's the scalable variant `mode="numeric"`.

## 2. Textbook sanity checks

**(a) One pool.** $\dot x = -kx$, $y = x$ — $k$ is recoverable ($k = -\dot y/y$). Identifiable.

**(b) Bellman & Åström (1970), the canonical *un*identifiable model.** $\dot x = -(k_1+k_2)x$,
$y = x$ — the output only ever sees the *sum* $k_1+k_2$; $k_1$ and $k_2$ individually are not
recoverable, no matter how good the data. The check should return rank $1 < 2$ and a nullspace
direction $\propto \partial/\partial k_1 - \partial/\partial k_2$ (i.e. "you can trade $k_1$ for
$k_2$ keeping $k_1+k_2$ fixed without changing any output").

In [2]:
x1 = sp.symbols('x1', positive=True)
k, k1, k2 = sp.symbols('k k1 k2', positive=True)

report("one pool:  dx/dt = -k x,  y = x  (x0 known)",
       structural_identifiability([-k * x1], [x1], [x1], [k], x0_known=True, mode="symbolic"))
report("Bellman:   dx/dt = -(k1+k2) x,  y = x  (x0 known)",
       structural_identifiability([-(k1 + k2) * x1], [x1], [x1], [k1, k2], x0_known=True, mode="symbolic"))
report("Bellman:   dx/dt = -(k1+k2) x,  y = x  (x0 UNKNOWN)",
       structural_identifiability([-(k1 + k2) * x1], [x1], [x1], [k1, k2], x0_known=False, mode="symbolic"))

one pool:  dx/dt = -k x,  y = x  (x0 known)      rank 1/1   IDENTIFIABLE
Bellman:   dx/dt = -(k1+k2) x,  y = x  (x0 known) rank 1/2   NOT identifiable   |  unresolved combination:  -1*k1 +1*k2   (only the complementary direction is pinned down)
Bellman:   dx/dt = -(k1+k2) x,  y = x  (x0 UNKNOWN) rank 2/3   NOT identifiable   |  unresolved combination:  -1*k1 +1*k2   (only the complementary direction is pinned down)


{'rank': 2,
 'full': 3,
 'identifiable': False,
 'null': [[0, -1, 1]],
 'params': ['x1', 'k1', 'k2']}

## 3. A gene circuit: negative autoregulation (from Lab 1)

$\dot x = \beta_0 + \beta\,/\,(1 + (x/K)^n) - \gamma x$, $y = x$ (you measure the protein).
Parameters $\theta = (\beta_0, \beta, K, \gamma)$. **We fix the Hill exponent $n = 2$** so the
model is rational — sympy's exact rank then works; treating $n$ as a free parameter introduces
$\log$ terms and is left to the exercises (and is where the *interesting* failure shows up — the
notorious $n$–$K$ confound). With $n$ fixed, measuring the protein turns out to be enough: the
circuit is structurally identifiable, with or without knowledge of $x_0$, and adding a second
output (a reporter of the production rate) doesn't change that. The lesson is the converse —
*when it fails, structural ID tells you exactly which extra measurement would fix it.*

In [3]:
b0, b, K, gam = sp.symbols('b0 b K gamma', positive=True)
n_hill = 2
hill_rep = lambda v: 1 / (1 + (v / K)**n_hill)
f_nar = b0 + b * hill_rep(x1) - gam * x1
prod_rate = b0 + b * hill_rep(x1)                       # an extra reporter, if you had one

report(f"NAR (Hill n={n_hill}):  y = x,           x0 known",
       structural_identifiability([f_nar], [x1], [x1], [b0, b, K, gam], x0_known=True))
report(f"NAR (Hill n={n_hill}):  y = x,           x0 UNKNOWN",
       structural_identifiability([f_nar], [x1], [x1], [b0, b, K, gam], x0_known=False))
report(f"NAR (Hill n={n_hill}):  y = (x, prod),   x0 UNKNOWN",
       structural_identifiability([f_nar], [x1, prod_rate], [x1], [b0, b, K, gam], x0_known=False))

NAR (Hill n=2):  y = x,           x0 known       rank 4/4   IDENTIFIABLE
NAR (Hill n=2):  y = x,           x0 UNKNOWN     rank 5/5   IDENTIFIABLE


NAR (Hill n=2):  y = (x, prod),   x0 UNKNOWN     rank 5/5   IDENTIFIABLE


{'rank': 5,
 'full': 5,
 'identifiable': True,
 'null': [],
 'params': ['x1', 'b0', 'b', 'K', 'gamma']}

## 4. The repressilator from a single output (from Lab 1)

$\dot p_i = \alpha\,/\,(1 + p_{i-1}^{\,2}) + \alpha_0 - \gamma p_i$ (Hill $n=2$ fixed, ring of 3),
$\theta = (\alpha, \alpha_0, \gamma)$. If you measure **only $p_1$** *and* know the initial state,
the parameters are structurally identifiable — the ring's asymmetric coupling ($p_1$ repressed by
$p_3$, $p_3$ by $p_2$, $p_2$ by $p_1$) lets the output derivatives back them out. If $x_0$ is also
unknown (6 unknowns from one output), the answer is more delicate — the check tells you. Measuring
all three proteins, of course, is fine. The takeaway: *what you measure* and *what you assume about
the initial state* both matter, and the rank condition makes "how much" precise. (The 6-unknown
case can take a minute or two — it stacks several Lie derivatives of a 3-vector rational field;
we use the numeric/random-substitution variant.)

In [4]:
p1, p2, p3 = sp.symbols('p1 p2 p3', positive=True)
alpha, alpha0, gam3 = sp.symbols('alpha alpha0 gamma', positive=True)
PERM = [2, 0, 1]; pp = [p1, p2, p3]
f_rep = [alpha / (1 + pp[PERM[i]]**2) + alpha0 - gam3 * pp[i] for i in range(3)]

report("repressilator:  y = p1,           x0 known",
       structural_identifiability(f_rep, [p1], [p1, p2, p3], [alpha, alpha0, gam3], x0_known=True))
report("repressilator:  y = p1,           x0 UNKNOWN",
       structural_identifiability(f_rep, [p1], [p1, p2, p3], [alpha, alpha0, gam3], x0_known=False))
report("repressilator:  y = (p1, p2, p3), x0 UNKNOWN",
       structural_identifiability(f_rep, [p1, p2, p3], [p1, p2, p3], [alpha, alpha0, gam3], x0_known=False))

repressilator:  y = p1,           x0 known       rank 3/3   IDENTIFIABLE


repressilator:  y = p1,           x0 UNKNOWN     rank 3/6   NOT identifiable   |  3 unresolved d.o.f. among {p1, p2, p3, alpha, alpha0, gamma}  (only 3 independent combinations are determined)


repressilator:  y = (p1, p2, p3), x0 UNKNOWN     rank 6/6   IDENTIFIABLE


{'rank': 6,
 'full': 6,
 'identifiable': True,
 'null': [],
 'params': ['p1', 'p2', 'p3', 'alpha', 'alpha0', 'gamma']}

## 5. For linear models, structural ID *is* the observability rank — the `jaxctrl` connection

When $f$ is linear ($\dot x = Ax$, $y = Cx$, parameters of $A$ aside), the Lie derivatives are just
$g = Cx$, $L_F g = CAx$, $L_F^2 g = CA^2 x$, …, so the rank condition above collapses to
$$
\operatorname{rank}\,\big[\,C;\ CA;\ CA^2;\ \dots;\ CA^{n-1}\,\big] = n,
$$
which is exactly **`jaxctrl.observability_matrix(A, C)`** / **`jaxctrl.is_observable(A, C)`** — the
numeric, linear special case of §1's symbolic, nonlinear machinery. Watch it on a small chain of
TFs (gene $i$ activates gene $i{+}1$): observing the *last* gene sees everything upstream;
observing the *first* sees nothing downstream.

This is the same rank that decided the network-control results of Lab 6. In
`scripts/benchmark_network_control.py` the linear surrogate of the regulome had its
controllability/observability Gramian *numerically singular on the master-regulator subspace* — i.e.
the surrogate's full $A$ is **not identifiable** from (and not controllable to) the master-regulator
outputs/inputs alone — which is precisely why the steer-to-target energy along the unreachable
directions came out unbounded, and why the SBI posterior over the GRN is correctly ridge-shaped
rather than a point.

In [5]:
# a TF activation chain: dx_i/dt = -x_i + x_{i-1}  (upstream activates downstream)
A = jnp.array([[-1., 0., 0., 0.],
               [ 1.,-1., 0., 0.],
               [ 0., 1.,-1., 0.],
               [ 0., 0., 1.,-1.]])
for label, C in [("observe x1 (most upstream)", jnp.array([[1., 0., 0., 0.]])),
                 ("observe x4 (most downstream)", jnp.array([[0., 0., 0., 1.]])),
                 ("observe x1 and x4",            jnp.array([[1., 0., 0., 0.], [0., 0., 0., 1.]]))]:
    rk = int(jnp.linalg.matrix_rank(jaxctrl.observability_matrix(A, C)))
    print(f"  {label:<30s} observability rank {rk}/4   ->  {'identifiable/observable' if rk == 4 else 'NOT — that subspace is invisible'}")

# tie back to Lab 6's network-control result, if it has been run
ncf = None
for base in [Path.cwd(), *Path.cwd().parents]:
    cand = base / "figures" / "network_control_results.json"
    if cand.exists(): ncf = json.loads(cand.read_text()); break
if ncf and ncf.get("controllability"):
    ct = ncf["controllability"]
    print(f"\n  benchmark_network_control.py:  the {ct.get('n_drivers','?')} master-regulator inputs reach a")
    print(f"  controllability subspace of rank only {ct.get('key_tf_set_rank','?')}/{ct.get('n','?')} of the linear surrogate")
    if ncf.get("steer_to_target", {}).get("gramian_min_eig_drivers") is not None:
        print(f"  (Gramian λ_min on that subspace ≈ {ncf['steer_to_target']['gramian_min_eig_drivers']:.1e} — numerically singular),")
    print("  ⇒ the surrogate's full A is not identifiable/controllable from the master regulators alone;")
    print("    that is the same fact, read in the dual direction, as the SBI posterior being ridge-shaped.")
else:
    print("\n  (run scripts/benchmark_network_control.py for the regulome-scale version of this story.)")

  observe x1 (most upstream)     observability rank 1/4   ->  NOT — that subspace is invisible
  observe x4 (most downstream)   observability rank 4/4   ->  identifiable/observable
  observe x1 and x4              observability rank 4/4   ->  identifiable/observable

  benchmark_network_control.py:  the 7 master-regulator inputs reach a
  controllability subspace of rank only 9/200 of the linear surrogate
  ⇒ the surrogate's full A is not identifiable/controllable from the master regulators alone;
    that is the same fact, read in the dual direction, as the SBI posterior being ridge-shaped.


## 6. Exercises

**(a) Free the Hill exponent.** In the NAR model, treat $n$ as an unknown parameter (alongside
$\beta_0,\beta,K,\gamma$). The symbolic rank now involves $\log$ terms and is heavy — use
`mode="numeric"` (random substitution). You should find the model becomes rank-deficient, and the
nullspace names the $n$–$K$ confound: from a single trajectory you can't separate "how cooperative"
from "what threshold". (This is the textbook reason Hill exponents are so hard to pin down — and why
COMBOS and friends matter for circuits with free $n$.)

**(b) The IRMA GRN.** Take the 5-gene IRMA-topology Hill-ODE from
`jaxctrl/examples/irma_sindy_lqr.ipynb` (Hill $n=2$ fixed). With all 5 species measured, is it
identifiable? Now drop to measuring 2 species, then 1 — find the *minimal output set* for
identifiability, and for the smaller sets read off the unidentifiable combinations.

**(c) Reproduce a COMBOS example.** Pick a 2- or 3-compartment model from the COMBOS / DiStefano
literature, run the check, and confirm the identifiable parameter combinations match what they
report. (Good cases: a 2-pool model with bidirectional exchange, observed from one pool.)

**(d) Close the loop with SBI.** Take one *unidentifiable* model (e.g. Bellman, or NAR with free $n$),
simulate noisy data, fit it with `optax` through `diffrax`, and walk the loss along the unidentifiable
nullspace direction — you'll find a flat valley. That's the *a posteriori* confirmation of the *a
priori* prediction, and it's the bridge to Lab 8 (the SBI / CellFlow inverse).

A starter for (a):

In [6]:
# --- Exercise (a) starter: NAR with the Hill exponent n as a free parameter ---
nH = sp.symbols('n', positive=True)
hill_rep_n = 1 / (1 + (x1 / K)**nH)
f_nar_n = b0 + b * hill_rep_n - gam * x1

# symbolic rank with log-terms is slow; use the random-substitution numeric variant
r = structural_identifiability([f_nar_n], [x1], [x1], [b0, b, K, gam, nH],
                               x0_known=True, mode="numeric", order=6)
report("NAR with FREE Hill exponent n:  y = x,  x0 known", r)
if not r["identifiable"]:
    print("\n  unresolved direction(s) over (b0, b, K, gamma, n):")
    for v in r["null"]:
        print("   ", "  ".join(f"{float(c):+.3g}·{p}" for c, p in zip(v, r["params"]) if abs(float(c)) > 1e-3))
    print("  ↳ interpret: which parameters can you only see in combination?  (look for K and n entangled.)")

# your turn: (b) the IRMA GRN, (c) a COMBOS compartmental example, (d) the SBI loss-valley.

NAR with FREE Hill exponent n:  y = x,  x0 known rank 3/5   NOT identifiable   |  2 unresolved d.o.f. among {b0, b, K, gamma, n}  (only 3 independent combinations are determined)

  unresolved direction(s) over (b0, b, K, gamma, n):
    +0.704·b0  -0.704·b  +0.003·K  -0.097·n
    -0.069·b0  +0.069·b  +0.044·K  -0.994·n
  ↳ interpret: which parameters can you only see in combination?  (look for K and n entangled.)


## Recap & where this sits

Three rungs of "identifiability," now all in your hands:

- **a priori — structural** (this lab): symbolic, model-only, "*could* you ever recover the
  parameters?" — the Taylor-series / observability-rank test (`structural_identifiability`), the
  symbolic generalisation of `jaxctrl`'s linear `is_observable` / `observability_gramian`. Run it
  *before* fitting any mechanistic reduced model — the Hill-ODE circuits of Lab 1, the IRMA GRN,
  the linear surrogate that warm-starts the anatomical compiler in Lab 8.
- **a posteriori — practical** (Lab 8): finite, noisy data — profile likelihoods, Fisher
  information, the width and ridges of the SBI / CellFlow posterior. When (a) says "not
  identifiable," (b) *shows* it as a flat valley.
- **structural ≠ module** (Lab 4): the Hodge-Laplacian Module Identifiability Index is a different
  beast — it asks whether the *network* decomposes into modules, not whether a *parameter vector* is
  recoverable. Don't conflate them (the whitepaper §2 spells this out).

*For free Hill exponents, larger circuits, or guaranteed-global answers, hand off to the dedicated
tools:* **COMBOS** (DiStefano lab — differential algebra, identifiable parameter combinations —
[biocyb0.cs.ucla.edu](http://biocyb0.cs.ucla.edu/wp/)), **SIAN**, **DAISY**, **GenSSI**,
**StructuralIdentifiability.jl**.

*References:* Bellman R, {\AA}str{\"o}m KJ, "On structural identifiability," *Math Biosci* 7:329–339
(1970); DiStefano III J, *Dynamic Systems Biology Modeling and Simulation*, Academic Press (2014);
Meshkat N, Kuo CE, DiStefano III J, "On finding and using identifiable parameter combinations …
and COMBOS," *PLoS ONE* 9(10):e110261 (2014). And `jaxctrl`'s `is_observable` / `observability_gramian`
for the numeric-linear version (Lab 6).